# Distillation Preprocessing (2)

What we do:
1) Generate user commands

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [ ]:
from core.types import *
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.utils.doom_game_state import DoomGameState
from pathlib import Path
from openai.types.responses import Response as OpenAIResponse

import json

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the pre-processed game states
game_states = GameStateEntry.load_states(
    path=Path("data/gamestates/perturbated-gamestates.json"),
    cls=DoomGameState
)

In [ ]:
openai_client = OpenAIClient[GameStateEntry, list[UserCommandEntry]](
    model="gpt-5.1",
    mode=ProcessingMode.BATCH,
    max_output_tokens=500,
    temperature=1.0,
    reasoning_effort='none',
    working_dir=Path('data/openai'),
)

In [ ]:
# Prepare Prompt
base_prompt = Path("prompts/command-generation-template.md").read_text(encoding="utf-8")
doom_root = Path("prompts/ultimate_doom")

full_prompt = base_prompt
for md_file in doom_root.rglob("*.md"):
    rel = md_file.relative_to(doom_root).with_suffix("")
    tag = "_".join(part.upper() for part in rel.parts)
    value = md_file.read_text(encoding="utf-8")
    full_prompt = full_prompt.replace(f"<{tag}>", value)

In [ ]:
print(full_prompt)
print(game_states[0].state.to_prompt_ready())

In [ ]:
def format_input(gse: GameStateEntry) -> str:
    return gse.state.to_prompt_ready()


def parse_output(response: OpenAIResponse, input_id: str, latency: float) -> list[UserCommandEntry]:
    try:
        raw_result = '\n'.join([
            resp.text
            for out in response.output
            for resp in out.content
            if out.type == 'message' and out.role == 'assistant'
            if resp.type == 'output_text'
        ])
        entries = list()
        for idx, line in enumerate(raw_result.splitlines()):
            data = json.loads(line)
            entries.append(
                UserCommandEntry(
                    id=f"{input_id}-uc{idx}",
                    state_id=input_id,
                    latency=latency,
                    command=UserCommand(**data)
                )
            )
        return entries
    except Exception as e:
        print(f"Invalid output: {e}")
        return []


def get_id(gse: GameStateEntry, idx: int) -> str:
    return gse.id

In [ ]:
outputs = openai_client.process(
    dataset=game_states,
    system_prompt=full_prompt,
    tools=[],
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
    batch_size=320,
)

In [ ]:
commands = [
    command
    for out in outputs
    for command in out
]

print(f"{len(commands)} user commands generated")

In [ ]:
# Save the current dataset
UserCommandEntry.save_commands(
    x=commands,
    path=Path("data/usercommands/user-commands.json")
)